In [1]:
import os
from dotenv import load_dotenv
load_dotenv(".env")
from smolagents import OpenAIModel,ChatMessage
gemini_key = os.environ.get("GEMINI_API_KEYS")

modelx = OpenAIModel(
    model_id="gemini-2.5-flash-lite",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=gemini_key
)

res = modelx.generate([
    ChatMessage(role="user", content="whats binary files")
])
res

ChatMessage(role='assistant', content='Binary files are computer files that store data in a non-human-readable format. Unlike text files, which store characters that represent letters, numbers, and symbols, binary files store data as sequences of **bits**, which are the fundamental units of information in computing (0s and 1s).\n\nHere\'s a breakdown of what that means and why they are important:\n\n**Key Characteristics of Binary Files:**\n\n*   **Raw Data:** They contain raw, unprocessed data in a format that is directly interpreted by a computer program or hardware.\n*   **Non-Human-Readable:** If you try to open a binary file in a standard text editor (like Notepad on Windows or TextEdit on macOS), you\'ll likely see a jumble of seemingly random characters, control codes, and potentially some recognizable text if there are embedded strings. This is because the editor is trying to interpret the raw binary data as characters, which it\'s not designed to do.\n*   **Specific Structure:

In [12]:
from io import BytesIO
from time import sleep

import helium
from dotenv import load_dotenv
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from smolagents import CodeAgent, tool
from smolagents.agents import ActionStep

# Load environment variables
load_dotenv()

@tool
def search_item_ctrl_f(text: str, nth_result: int = 1) -> str:
    """
    Searches for text on the current page via Ctrl + F and jumps to the nth occurrence.
    Args:
        text: The text to search for
        nth_result: Which occurrence to jump to (default: 1)
    """
    elements = driver.find_elements(By.XPATH, f"//*[contains(text(), '{text}')]")
    if nth_result > len(elements):
        raise Exception(f"Match n°{nth_result} not found (only {len(elements)} matches found)")
    result = f"Found {len(elements)} matches for '{text}'."
    elem = elements[nth_result - 1]
    driver.execute_script("arguments[0].scrollIntoView(true);", elem)
    result += f"Focused on element {nth_result} of {len(elements)}"
    return result

@tool
def go_back() -> None:
    """Goes back to previous page."""
    driver.back()

@tool
def close_popups() -> str:
    """
    Closes any visible modal or pop-up on the page. Use this to dismiss pop-up windows!
    This does not work on cookie consent banners.
    """
    webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()

# Configure Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--force-device-scale-factor=1")
chrome_options.add_argument("--window-size=1000,1350")
chrome_options.add_argument("--disable-pdf-viewer")
chrome_options.add_argument("--window-position=0,0")

# Initialize the browser
try:
    helium.kill_browser()
except:
    pass
driver = helium.start_chrome(headless=False, options=chrome_options)

# Set up screenshot callback
def save_screenshot(memory_step: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0)  # Let JavaScript animations happen before taking the screenshot
    driver = helium.get_driver()
    current_step = memory_step.step_number
    if driver is not None:
        for previous_memory_step in agent.memory.steps:  # Remove previous screenshots for lean processing
            if isinstance(previous_memory_step, ActionStep) and previous_memory_step.step_number <= current_step - 2:
                previous_memory_step.observations_images = None
        png_bytes = driver.get_screenshot_as_png()
        image = Image.open(BytesIO(png_bytes))
        print(f"Captured a browser screenshot: {image.size} pixels")
        memory_step.observations_images = [image.copy()]  # Create a copy to ensure it persists

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    memory_step.observations = (
        url_info if memory_step.observations is None else memory_step.observations + "\n" + url_info
    )


helium_instructions = """
You can use helium to access websites. Don't bother about the helium driver, it's already managed.
We've already ran "from helium import *"
Then you can go to pages!
Code:
go_to('github.com/trending')
```<end_code>

You can directly click clickable elements by inputting the text that appears on them.
Code:
click("Top products")
```<end_code>

If it's a link:
Code:
click(Link("Top products"))
```<end_code>

If you try to interact with an element and it's not found, you'll get a LookupError.
In general stop your action after each button click to see what happens on your screenshot.
Never try to login in a page.

To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.
Code:
scroll_down(num_pixels=1200) # This will scroll one viewport down
```<end_code>

When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or targeting an 'X' element (this most often fails).
Just use your built-in tool `close_popups` to close them:
Code:
close_popups()
```<end_code>

You can use .exists() to check for the existence of an element. For example:
Code:
if Text('Accept cookies?').exists():
    click('I accept')
```<end_code>
"""

In [13]:
from smolagents import InferenceClientModel
# Create the agent
agent = CodeAgent(
    tools=[go_back, close_popups, search_item_ctrl_f],
    model=modelx,
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],
    max_steps=20,
    verbosity_level=2,
)

# Import helium for the agent
agent.python_executor("from helium import *")

CodeOutput(output=None, logs='', is_final_answer=False)

In [ ]:
github_request = """
I'm trying to find how hard I have to work to get a repo in github.com/trending.
Can you navigate to the profile for the top author of the top trending repo, and give me their total number of commits over the last year?
"""

agent_output = agent.run(github_request + helium_instructions)
print("Final output:")
print(agent_output)

In [ ]:
import helium
from selenium import webdriver
from time import sleep

# 1. Configure Browser (Optional, but makes it look better)
options = webdriver.ChromeOptions()
options.add_argument("--window-size=1200,900")

# 2. Open Chrome and go to Google
print("Opening Chrome...")
helium.start_chrome("https://www.google.com", options=options)

# 3. Handle Google's search bar
# Helium is smart: it looks for a text field to 'write' into
print("indian stock market summary of today")
helium.write("indian stock market summary of today")
helium.press(helium.ENTER)

# 4. Find the correct link and click it
# We wait 2 seconds for the page to load
sleep(2)
if helium.Text("Trending repositories on GitHub").exists():
    helium.click("Trending repositories on GitHub")
    print("Success! Navigated to GitHub Trending.")
else:
    # Fallback: Just click the first link that mentions github

    helium.click("I am not a robot checkbox")

# Keep the browser open for 10 seconds so you can see the result
sleep(10)
helium.kill_browser()

In [7]:
import undetected_chromedriver as uc
import helium
from time import sleep

# Use undetected_chromedriver instead of standard selenium
options = uc.ChromeOptions()
driver = uc.Chrome(version_main=145,options=options)

# Link helium to this "stealth" driver
helium.set_driver(driver)

helium.go_to("https://www.google.com")
helium.write("indian stock market summary of today")
helium.press(helium.ENTER)
sleep(2)

if helium.S("h3").exists():
    helium.click(helium.S("h3"))
else:
    print("could'nt resolve")
# Now it's much less likely to trigger the Robot check

In [12]:
from helium import *
import undetected_chromedriver as uc
from time import sleep

# Setup
options = uc.ChromeOptions()
driver = uc.Chrome(version_main=145, options=options)
set_driver(driver)

go_to("https://www.google.com")
write("Cyberpunk Aesthetic 4k")
press(ENTER)

# --- THE "COOL" PART ---

# 1. Wait for 'Images' to actually appear (up to 10 seconds)
wait_until(Text("Images").exists)

if Text("Images").exists():
    print("Found it! Clicking the Images tab...")
    click("Images")
    
    # 2. Let's do something even cooler: Click the 3rd image 
    # Helium doesn't just click text; it can click 'S'electors too
    sleep(2)
    # Most Google images have a specific class or are just <img> tags
    images = find_all(S("img"))
    if len(images) > 5:
        click(images[5]) # Clicks the 6th image found
        print("Opened a specific image!")

sleep(5)
kill_browser()

Found it! Clicking the Images tab...
Opened a specific image!


In [14]:
import undetected_chromedriver as uc
import helium
from time import sleep

# 1. Setup (using your working version)
options = uc.ChromeOptions()
driver = uc.Chrome(version_main=145, options=options)
helium.set_driver(driver)

try:

    # --- TEST 2: Drag and Drop (The Hard Stuff) ---
    # print("Test 2: Handling complex UI (Drag & Drop)...")
    # helium.go_to("https://bootsnipp.com/snippets/featured/simple-drag-and-drop-list")
    
    # # In standard Selenium, drag-and-drop is 10+ lines of code. 
    # # In Helium, it's one line.
    # # Note: We use S() to find the list items
    # sleep(3)
    # print("Simulating a 'Done' list...")
    # # Drag the first item to the bottom (conceptual demo)
    # sleep(3)
    # helium.drag("List item 1", to="List item 5") 

    # # --- TEST 3: Relative Positioning ---
    print("Test 3: Clicking things based on where they are...")
    helium.go_to("https://www.wikipedia.org/")
    
    # Tell Helium to click the search bar, but specifically the one 
    # that is 'below' the Wikipedia Logo
    helium.click(helium.S("#searchInput"))
    helium.write("Python (programming language)")
    helium.press(helium.ENTER)
    sleep(3)
    # Scrape the first paragraph easily
    first_paragraph = helium.find_all(helium.S("p"))[0].web_element.text
    print(f"\nResult from Wikipedia:\n{first_paragraph[:200]}...")

    print("\nScript complete! Helium handled everything.")
    sleep(5)

finally:
    helium.kill_browser()

Test 3: Clicking things based on where they are...

Result from Wikipedia:
Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation.[38] Python is dynamically type-checked and garb...

Script complete! Helium handled everything.
